<a href="https://colab.research.google.com/github/Jyotsna135-bit/GenerativeAI/blob/main/GenerativeAI/Notebooks/Deployment/Ollama/ollama_embedding_gemma4_documented.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Embedding Model + LLM Deployment using Ollama (Gemma 4)

This builds on the previous Ollama notebooks. On top of running an LLM, we're now also running an **embedding model** through the same Ollama server.

Embeddings are how you turn text into numbers. The model reads a sentence and outputs a vector — a fixed-length list of floats that represents the meaning of that text. Two sentences that mean similar things end up with vectors that are close to each other. That's what makes semantic search and RAG systems work.

**Models:**
- Embedding: `nomic-embed-text` — open source from Nomic AI, outputs 768-dimensional vectors
- LLM: `gemma3:4b` — Google's Gemma 4, 4B parameters, noticeably better than Gemma 2 at following instructions

The point of having both in one notebook is to show a real RAG flow end to end — embed some documents, find the relevant one, pass it to the LLM.

> Set runtime to **T4 GPU** before running.

## Installing Ollama

`zstd` is needed by the Ollama installer to unpack itself — install it first or the setup fails partway through.

In [1]:
!apt-get install -y zstd -qq
!curl -fsSL https://ollama.com/install.sh | sh

Selecting previously unselected package zstd.
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


## Pulling the Models

Both models get pulled upfront. Ollama treats them like separate downloadable packages — same server serves both, you just switch model names in the request.

In [2]:
import subprocess, time, requests

subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(15):
    try:
        if requests.get("http://localhost:11434").status_code == 200:
            print("ollama is running")
            break
    except:
        time.sleep(1)

ollama is running


## Semantic Similarity

This is the simplest way to show embeddings are doing something useful. We take three sentences — two about AI, one about pizza — and measure how close their vectors are using cosine similarity.

The score goes from 0 to 1. The two AI sentences should score much higher than the pizza comparison, even though none of them share exact words. That's the whole point — the model understands meaning, not just keywords.

In [3]:
# embedding model — converts text to vectors
!ollama pull nomic-embed-text

# LLM — generates text responses
!ollama pull gemma3:4b

## Batch Embedding

In practice you'd be embedding a lot of documents at once before storing them. This just shows the pattern — loop through texts, embed each one, store the vectors.

In [4]:
import requests, json

def get_embedding(text):
    response = requests.post(
        "http://localhost:11434/api/embeddings",
        json={"model": "nomic-embed-text", "prompt": text}
    )
    return response.json()["embedding"]

embedding = get_embedding("What is a large language model?")
print(f"Embedding dimensions: {len(embedding)}")
print(f"First 5 values: {embedding[:5]}")

Embedding dimensions: 768
First 5 values: [-1.1624970436096191, 1.9516496658325195, -3.4593796730041504, -1.9797314405441284, 0.10939395427703857]


## Semantic Similarity with Embeddings

To show embeddings are useful, we compare three sentences using **cosine similarity** — a measure of how similar two vectors are (1.0 = identical meaning, 0.0 = completely different).

You'll see that the two sentences about AI score much higher than the sentence about pizza, even though none of them share exact words.

In [5]:
import numpy as np

def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

sentences = [
    "Large language models are trained on massive text datasets.",
    "Neural networks learn patterns from large amounts of data.",
    "Pizza is a popular dish made with dough, sauce, and cheese."
]

embeddings = [get_embedding(s) for s in sentences]

print("Similarity: AI sentence 1 vs AI sentence 2:", round(cosine_similarity(embeddings[0], embeddings[1]), 4))
print("Similarity: AI sentence 1 vs Pizza sentence:", round(cosine_similarity(embeddings[0], embeddings[2]), 4))

Similarity: AI sentence 1 vs AI sentence 2: 0.6983
Similarity: AI sentence 1 vs Pizza sentence: 0.4287


## Simple RAG Pipeline

This is a minimal RAG (Retrieval-Augmented Generation) pipeline:

1. We have a small knowledge base of 4 documents
2. We embed the user's question and all documents
3. We find the most similar document using cosine similarity
4. We pass that document as context to the LLM to generate a grounded answer

This is exactly how production RAG systems work — just at a much larger scale.

In [6]:
from openai import OpenAI

!pip install openai -q

llm_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",
)

# small knowledge base
documents = [
    "Ollama is a tool for running large language models locally on your machine without an internet connection.",
    "vLLM is a high-throughput inference engine that uses PagedAttention to efficiently manage GPU memory.",
    "Gemma is an open-source language model created by Google DeepMind, available in 2B and 7B parameter sizes.",
    "RAG stands for Retrieval-Augmented Generation. It combines a retrieval system with a language model to answer questions using external documents."
]

# embed all documents
doc_embeddings = [get_embedding(doc) for doc in documents]

def rag_answer(question):
    # embed the question
    q_embedding = get_embedding(question)

    # find the most relevant document
    similarities = [cosine_similarity(q_embedding, doc_emb) for doc_emb in doc_embeddings]
    best_idx = int(np.argmax(similarities))
    best_doc = documents[best_idx]

    print(f"Most relevant document (similarity={similarities[best_idx]:.4f}):")
    print(f"  → {best_doc}\n")

    # ask the LLM using the retrieved document as context
    prompt = f"Use the following context to answer the question.\n\nContext: {best_doc}\n\nQuestion: {question}"
    response = llm_client.chat.completions.create(
        model="gemma3:4b",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=200
    )
    return response.choices[0].message.content

answer = rag_answer("What is vLLM and what makes it efficient?")
print("Answer:", answer)

Most relevant document (similarity=0.7815):
  → vLLM is a high-throughput inference engine that uses PagedAttention to efficiently manage GPU memory.

Answer: According to the provided text, **vLLM is a high-throughput inference engine** that utilizes **PagedAttention** to efficiently manage GPU memory. 

Essentially, PagedAttention allows vLLM to handle large language models more effectively by optimizing how GPU memory is used.


## Batch Embedding

In practice you'd often need to embed many documents at once — for example when indexing a dataset. Here we embed a list of sentences in a loop and store them, which is the pattern you'd use before building a vector database.

In [7]:
texts = [
    "Ollama makes local LLM deployment easy.",
    "vLLM is optimised for production serving.",
    "Embeddings capture semantic meaning of text.",
    "RAG improves LLM answers using retrieved context."
]

batch_embeddings = [get_embedding(t) for t in texts]

for text, emb in zip(texts, batch_embeddings):
    print(f"'{text[:50]}' → vector of length {len(emb)}")

'Ollama makes local LLM deployment easy.' → vector of length 768
'vLLM is optimised for production serving.' → vector of length 768
'Embeddings capture semantic meaning of text.' → vector of length 768
'RAG improves LLM answers using retrieved context.' → vector of length 768


## Summary

| Feature | Details |
|---|---|
| Embedding model | `nomic-embed-text` via Ollama |
| LLM | `gemma3:4b` via Ollama |
| Embedding endpoint | `POST /api/embeddings` |
| LLM endpoint | `POST /v1/chat/completions` (OpenAI-compatible) |
| Use case demonstrated | Semantic similarity + simple RAG pipeline |

Both models run through the same Ollama server — no extra infrastructure needed. This is the main advantage of Ollama for local development: you can prototype a full RAG system without any cloud services.